In [ ]:
# 1) Скачать репозиторий
from pathlib import Path

REPO_URL = 'https://github.com/AnixGG/Mot-flow-research.git' 
REPO_DIR = Path('/content/MOT')

if not (REPO_DIR / '.git').exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd /content/MOT
!git -C /content/MOT rev-parse HEAD

In [ ]:
# 2) Скачать/зафиксировать версию ultralytics
ULTRALYTICS_REPO_URL = 'https://github.com/ultralytics/ultralytics.git'
ULTRALYTICS_REF = 'f9dec228e746159a97a6868878ab25abc8c81846'  # из env_info.txt
ULTRALYTICS_DIR = Path('/content/MOT/ultralytics')

if not (ULTRALYTICS_DIR / '.git').exists():
    !git clone {ULTRALYTICS_REPO_URL} {ULTRALYTICS_DIR}

!git -C {ULTRALYTICS_DIR} fetch --all --tags
!git -C {ULTRALYTICS_DIR} checkout {ULTRALYTICS_REF}
!git -C {ULTRALYTICS_DIR} rev-parse HEAD

In [ ]:
# 3) Скачать MOT17 по ссылке (и альтернатива через Google Drive ниже в комментариях)
MOT17_URL = 'https://motchallenge.net/data/MOT17.zip' 
DATASET_ROOT = Path('/content/MOT/dataset')
ARCHIVE_PATH = DATASET_ROOT / 'MOT17.zip'

DATASET_ROOT.mkdir(parents=True, exist_ok=True)
!wget -c {MOT17_URL} -O {ARCHIVE_PATH}
!unzip -oq {ARCHIVE_PATH} -d {DATASET_ROOT}
!ls -la {DATASET_ROOT / 'MOT17'}

# ---- Альтернатива Google Drive ----
# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/drive/MyDrive/datasets/MOT17.zip {ARCHIVE_PATH}
# !unzip -oq {ARCHIVE_PATH} -d {DATASET_ROOT}


In [ ]:
# 4) Установить Python-зависимости и библиотеки
# Python в Colab уже предустановлен; эта ячейка ставит пакеты окружения.
!python --version
!nvidia-smi
!python -m pip install -U pip setuptools wheel
!python -m pip install numpy pandas PyYAML motmetrics opencv-python
!python -m pip install -r /content/MOT/ultralytics/requirements.txt
!python -m pip install -e /content/MOT/ultralytics

In [ ]:
# 5) Скачать yolo11n.pt 
YOLO11N_URL = 'https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11n.pt'
YOLO11N_PATH = Path('/content/MOT/models/detector/yolo11n.pt')

YOLO11N_PATH.parent.mkdir(parents=True, exist_ok=True)
!wget -c {YOLO11N_URL} -O {YOLO11N_PATH}
!ls -lh {YOLO11N_PATH}

In [ ]:
import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
import yaml
from pathlib import Path


# DATA_ROOT = Path('/content/drive/MyDrive/datasets/MOT17/train')
DATA_ROOT = Path('/content/MOT/dataset/MOT17/train')
SEQUENCES = ['MOT17-02-FRCNN', 'MOT17-04-FRCNN']
OUTDIR = Path('/content/MOT/baseline_runs/mot17_botsort_colab')

assert DATA_ROOT.exists(), f'Missing DATA_ROOT: {DATA_ROOT}'
for seq in SEQUENCES:
    seq_dir = DATA_ROOT / seq
    assert (seq_dir / 'img1').exists(), f'Missing frames: {seq_dir / "img1"}'
    assert (seq_dir / 'gt' / 'gt.txt').exists(), f'Missing GT: {seq_dir / "gt/gt.txt"}'

run_cfg = {
    'data': str(DATA_ROOT),
    'sequences': SEQUENCES,
    'outdir': str(OUTDIR),
    'device': '0',
    'model': 'yolo11n.pt',
    'conf': 0.25,
    'iou': 0.7,
    'imgsz': 1280,
    'classes': [0],
    'tracker_config': '/content/MOT/configs/botsort_baseline.yaml',
    'visualization': {
        'enabled': False,
        'fps': 30
    }
}

cfg_path = Path('/content/MOT/configs/run_baseline_colab.yaml')
with cfg_path.open('w', encoding='utf-8') as f:
    yaml.safe_dump(run_cfg, f, sort_keys=False, allow_unicode=False)

print('Config written to:', cfg_path)
print(cfg_path.read_text(encoding='utf-8'))

In [ ]:
%cd /content/MOT
!python scripts/run_baseline.py

In [ ]:
from pathlib import Path

OUTDIR = Path('/content/MOT/baseline_runs/mot17_botsort_colab')
print('Metrics file:', OUTDIR / 'metrics.csv')
print((OUTDIR / 'metrics.csv').read_text(encoding='utf-8'))
print('Timing file:', OUTDIR / 'timing.csv')
print((OUTDIR / 'timing.csv').read_text(encoding='utf-8'))

In [ ]:
# Optional: сохранить результаты в Google Drive
from pathlib import Path

OUTDIR = Path('/content/MOT/baseline_runs/mot17_botsort_colab')
DRIVE_EXPORT = Path('/content/drive/MyDrive/MOT_runs/mot17_botsort_colab')
DRIVE_EXPORT.parent.mkdir(parents=True, exist_ok=True)
!cp -r {OUTDIR} {DRIVE_EXPORT}
print('Copied to:', DRIVE_EXPORT)